# 04 — ML Model Development: Part 1 (Price & Value Retention Analysis)

This notebook develops machine learning models for predicting Steam game pricing outcomes.

The goal is to model two related problems:

- **[Regression]** Predict `current_price` — the current price of a game in PHP.
- **[Classification]** Predict `value_retention_tier` — whether a game keeps its launch value or becomes heavily discounted over time.

Data source: `cleaned_games` table from the cleaned Steam database, which combines Steam Storefront, SteamSpy, review, pricing, and player engagement features into one analysis-ready dataset.

ML problems:

- **[Regression]** Predict `current_price` using game age, launch price, reviews, ownership, achievements, genre, developer tier, multiplayer status, and price tier.
- **[Classification]** Predict `value_retention_tier` based on the ratio between `current_price` and `initial_price`.

Steps:

1. Load and inspect the data
2. Define targets (Step 4.1)
3. Select features and preprocess data (Step 4.3)
4. Train/test split (Step 4.2)
5. Regression models for current price prediction (Step 4.4)
6. Classification models for value retention tier prediction (Step 4.5)
7. Model comparison and feature importance

## Setup

In [48]:
import sys
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingRegressor,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, label_binarize, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight

# Bayesian hyperparameter tuning (TPE sampler)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [49]:
import sqlite3
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DB_PATH = PROJECT_ROOT / "cleaned data" / "steam.db"

assert DB_PATH.exists(), f"DB not found at: {DB_PATH}"

conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("SELECT * FROM cleaned_games", conn)

print(f"Connected to: {DB_PATH}")
print(f"Loaded {len(df):,} rows from cleaned_games")

Connected to: /Users/faithtiukinhoy/Documents/GitHub/Y2T2-Final-Project/cleaned data/steam.db
Loaded 4,924 rows from cleaned_games


In [84]:
# =====================================================================
# Tuning cache toggle
# =====================================================================
# Optuna tuning takes ~50 min total across the four model-tuning cells
# (rf_tune, gbm_tune, step45_train, panel_tune). Setting RETRAIN_MODELS = False
# loads previously-tuned models from `outputs/models/*.joblib` instead of
# re-running the search -- the notebook re-runs in seconds.
#
# Set RETRAIN_MODELS = True after:
#   - cleaned_games / cleaned_discount_panel changes (rerun nb 02)
#   - feature-list changes (NUM_FEATURES, panel_num_features, etc.)
#   - any change to the preprocessor or target derivation
#   - simply when you want to refresh the cache
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RETRAIN_MODELS = True

MODELS_CACHE_DIR = PROJECT_ROOT / "outputs" / "models"
MODELS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'RETRAIN_MODELS = {RETRAIN_MODELS}')
print(f'Cache directory: {MODELS_CACHE_DIR}')

if not RETRAIN_MODELS:
    cached = sorted(MODELS_CACHE_DIR.glob('*.joblib'))
    print(f'Existing cached models ({len(cached)}):')
    
    for p in cached:
        size_kb = p.stat().st_size / 1024
        print(f'  {p.name:<28} {size_kb:>7.1f} KB')

RETRAIN_MODELS = True
Cache directory: /Users/faithtiukinhoy/Documents/GitHub/Y2T2-Final-Project/outputs/models


In [51]:
# =====================================================================
# Auto-save figures
# =====================================================================
# Saves every figure to outputs/figures/nb04/ with a sequential prefix +
# title-derived filename, e.g. `001_regression_target_distribution.png`.
#
# Implementation: wraps `plt.show()` so figures get saved BEFORE matplotlib's
# inline backend closes them. (An earlier post_run_cell-hook version fired
# too late -- matplotlib_inline closes figures during its own post_execute
# callback that runs before post_run_cell.)
#
# Set SAVE_FIGURES = False to disable.
import re as _re
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SAVE_FIGURES = True
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures" / "nb04"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = [0]

_orig_plt_show = getattr(plt.show, "_orig_plt_show", plt.show)

def _save_then_show(*args, **kwargs):
    if SAVE_FIGURES:
        for num in plt.get_fignums():
            fig = plt.figure(num)
            title = next((ax.get_title().strip() for ax in fig.axes if ax.get_title().strip()), "")
            stem = _re.sub(r'[^a-z0-9_-]+', '_', title.lower()).strip('_')[:80] or 'untitled'
            _fig_counter[0] += 1
            path = FIGURES_DIR / f'{_fig_counter[0]:03d}_{stem}.png'
            fig.savefig(path, dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
    return _orig_plt_show(*args, **kwargs)

_save_then_show._orig_plt_show = _orig_plt_show
plt.show = _save_then_show

print(f'SAVE_FIGURES = {SAVE_FIGURES}')
print(f'plt.show wrapped; saving to: {FIGURES_DIR}/')

SAVE_FIGURES = True
plt.show wrapped; saving to: /Users/faithtiukinhoy/Documents/GitHub/Y2T2-Final-Project/outputs/figures/nb04/


> ## Step 4.1: Define Target Variables

Two complementary targets — the regression captures continuous depreciation, the classification captures discrete strategy archetypes.

### Regression target: `discount_depth` (continuous, 0–1)
Fraction of launch price the game has shed. A value of 0.5 means the game is currently selling at 50% off launch. Predicting this directly avoids the tautology of predicting absolute `current_price` (where `initial_price` would mechanically dominate feature importance at ~0.9, drowning out the predictors that actually matter).

### Classification target: `value_retention_tier` (4 classes)
Strict definitions so the labels capture genuine retention behavior, not just "recently released games that haven't gone on sale yet":

- **Premium Hold** — `value_ratio > 0.85` **AND** `age ≥ 1 year` **AND** `ever_discounted == 0` (the game has been on the market long enough AND has actively resisted discounting)
- **Standard Depreciation** — `value_ratio > 0.50`
- **Heavy Discount** — `value_ratio > 0.25`
- **Permanent Bargain** — `value_ratio ≤ 0.25`

Without the time + ever_discounted gates, every newly-released full-price game would qualify as Premium Hold and the class would mean nothing.

In [52]:
# Drop games with invalid launch price (can't compute a depreciation ratio)
df_model = df[df['initial_price'] > 0].copy()
print(f"After filtering initial_price > 0: {len(df_model):,} of {len(df):,} games kept")

# Log-transform heavily right-skewed numerics. All three end up in NUM_FEATURES
# in cell ccb410d0 in their log form so StandardScaler doesn't get dominated
# by the long-tail extremes (e.g. 16 achievement-farming games with > 1,000
# achievements, or premium $200+ titles).
df_model['log_total_reviews']      = np.log1p(df_model['total_reviews'])
df_model['log_initial_price']      = np.log1p(df_model['initial_price'])
df_model['log_achievements_total'] = np.log1p(df_model['achievements_total'])

# Regression target sanity check (already in cleaned_games)
print(f"\nRegression target — max_discount_ever / 100 (deepest historical discount, 0–1):")
print(f"  mean   : {(df_model['max_discount_ever'] / 100).mean():.3f}")
print(f"  median : {(df_model['max_discount_ever'] / 100).median():.3f}")
print(f"  max    : {(df_model['max_discount_ever'] / 100).max():.3f}")


After filtering initial_price > 0: 4,003 of 4,924 games kept

Regression target — max_discount_ever / 100 (deepest historical discount, 0–1):
  mean   : 0.722
  median : 0.800
  max    : 1.000


> ## Step 4.2: Feature Selection + Filter Funnel

**Numeric features (6):**
- `days_since_release` — age (fundamental driver of depreciation)
- `initial_price` — kept as a feature because the mechanical link to the target is broken now that we predict `discount_depth` (a ratio) rather than absolute price
- `review_score` — quality
- `log_total_reviews` — popularity (log-transformed for skew)
- `log_ownership` — owner count from SteamSpy bucket
- `achievements_total` — engagement structure

**Categorical features (5):**
- `primary_genre` — depreciation patterns differ by genre
- `is_multiplayer` — live-service games tend to hold value better
- `has_controller_support` — proxy for production quality
- `release_month` — captures seasonal launch effects (1–12, treated categorically)
- `player_engagement` — current-popularity signal (Low/Medium/High/Unknown)

The **filter funnel** below shows how many rows survive each step. If a step drops a lot of data, that's where to investigate.

In [53]:
# ============================================================
# Additional feature engineering: player drop-off
# ============================================================

# Use available columns from df_model:
# ownership_midpoint = estimated ownership
# avg_recent_players = recent SteamCharts player activity

df_model['initial_players'] = df_model['ownership_midpoint'] * 0.10

df_model['recent_players'] = df_model['avg_recent_players'].clip(lower=1)

df_model['player_dropoff'] = (
    df_model['initial_players'] - df_model['recent_players']
)

df_model['player_retention_ratio'] = (
    df_model['recent_players'] / (df_model['initial_players'] + 1)
)

df_model['log_player_dropoff'] = np.log1p(
    df_model['player_dropoff'].clip(lower=0)
)

print(df_model[['ownership_midpoint', 'avg_recent_players', 'initial_players',
                'recent_players', 'player_dropoff', 
                'player_retention_ratio', 'log_player_dropoff']].head())

   ownership_midpoint  avg_recent_players  initial_players  recent_players  \
0          15000000.0         8198.682008        1500000.0     8198.682008   
1           1500000.0           46.152022         150000.0       46.152022   
2           7500000.0           84.144847         750000.0       84.144847   
3           7500000.0            3.941392         750000.0        3.941392   
4           3500000.0           87.032033         350000.0       87.032033   

   player_dropoff  player_retention_ratio  log_player_dropoff  
0    1.491801e+06                0.005466           14.215496  
1    1.499538e+05                0.000308           11.918090  
2    7.499159e+05                0.000112           13.527718  
3    7.499961e+05                0.000005           13.527825  
4    3.499130e+05                0.000249           12.765443  


In [54]:
print(df_model.columns.tolist())

['appid', 'title', 'developer', 'publisher', 'release_date', 'days_since_release', 'release_year', 'release_month', 'is_legacy', 'initial_price', 'log_initial_price', 'current_price', 'price_tier', 'price_retention_ratio', 'review_score', 'total_reviews', 'log_total_reviews', 'total_positive', 'total_negative', 'ownership_midpoint', 'log_ownership', 'units_sold_estimate', 'primary_genre', 'is_multiplayer', 'has_controller_support', 'achievements_total', 'log_achievements_total', 'ever_discounted', 'max_discount_ever', 'player_engagement', 'avg_recent_players', 'has_itad_data', 'initial_players', 'recent_players', 'player_dropoff', 'player_retention_ratio', 'log_player_dropoff']


In [55]:
# Feature lists. NUM_FEATURES are scaled, CAT_FEATURES are one-hot encoded,
# PUB_FEATURE is target-encoded separately because publisher has thousands
# of unique values (high cardinality).
NUM_FEATURES = [
    'days_since_release',
    'log_initial_price',
    'review_score',
    'log_total_reviews',
    'log_ownership',
    'log_achievements_total',
    'log_player_dropoff',
    'player_retention_ratio'
]
CAT_FEATURES = [
    'primary_genre',
    'is_multiplayer',
    'has_controller_support',
    'release_month',
    'player_engagement',
]
PUB_FEATURE = ['publisher']

# Filter funnel - explicit row-count diagnostic so we know exactly what was dropped.
required = NUM_FEATURES + CAT_FEATURES + PUB_FEATURE + ['max_discount_ever']
print('Filter funnel:')
print(f'  Loaded from cleaned_games                : {len(df):>5,}')
print(f'  + initial_price > 0                      : {len(df_model):>5,}')
mask_features = df_model[required].notna().all(axis=1)
print(f'  + non-null required features and targets : {mask_features.sum():>5,}')
mask_observed = df_model['max_discount_ever'] > 0   # drop games never observed on sale (likely no ITAD coverage)
print(f'  + has observed sale history (cut > 0)    : {(mask_features & mask_observed).sum():>5,}')
df_clean = df_model[mask_features & mask_observed].copy()
print(f'\nFinal modelling dataset: {len(df_clean):,} games')


Filter funnel:
  Loaded from cleaned_games                : 4,924
  + initial_price > 0                      : 4,003
  + non-null required features and targets : 3,963
  + has observed sale history (cut > 0)    : 3,928

Final modelling dataset: 3,928 games


In [56]:
# Preprocessing pipeline -- three branches:
#   - StandardScaler on continuous numerics
#   - OneHotEncoder on low-cardinality categoricals (drop one column for binary
#     features so we don't carry redundant dummy pairs)
#   - TargetEncoder on `publisher` (high-cardinality; uses out-of-fold means
#     during fit so no leakage into the regression target)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                                                     NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'),             CAT_FEATURES),
    ('pub', TargetEncoder(smooth=10.0, target_type='auto', random_state=42),      PUB_FEATURE),
])
X_transformed = preprocessor.fit_transform(
    df_clean[NUM_FEATURES + CAT_FEATURES + PUB_FEATURE],
    df_clean['max_discount_ever'] / 100,
)
print(f'Transformed feature space: {X_transformed.shape[1]} columns after one-hot + target encoding')


Transformed feature space: 50 columns after one-hot + target encoding
